# 00 ? Quickstart

`QickworkspaceV2` is a self-contained automated quantum calibration framework built on QICK.
This notebook assumes a live QICK session.

What you will learn:
1. Start a QICK session
2. Build a hardware config
3. Run your first experiment
4. Inspect the `ExperimentData` result


In [ ]:
import sys
sys.path.insert(0, r'../')   # adjust if running from a different directory

import QickworkspaceV2
print('QickworkspaceV2', QickworkspaceV2.__version__)

## 1. Start a QICK session

`BaseExperiment.connect_pyro4()` connects to the QICK board and registers `soc` / `soccfg` globally for experiment classes.


In [ ]:
from QickworkspaceV2 import BaseExperiment

BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82',
    data_path=r'D:\Labber_Data\Jay\test',
)


## 2. Load the current hardware config

Hardware keys and channel assignments are maintained in `QickworkspaceV2/config/system_cfg.py`.
Use that config as the source of truth, then add only the sweep values required by this run.


In [ ]:
from qick.asm_v2 import QickSweep1D
from QickworkspaceV2 import ExperimentConfig
from QickworkspaceV2.config.system_cfg import config_list

qubit = 'Q1'
cfg_all = ExperimentConfig(config_list)
base_cfg = cfg_all.get_qubit(qubit)
center = base_cfg['res_freq_ge']

cfg = cfg_all.get_qubit(qubit)
cfg.update([
    ('steps', 101),
    ('res_freq_ge', QickSweep1D('freqloop', center - 10, center + 10)),
    ('relax_delay', 0),
])
print('Configured qubits:', cfg_all.qubit_names())


## 3. Run Resonator Spectroscopy

`ResonatorSpec` sweeps the resonator drive frequency and fits a circle (ABCD hanger model)
to extract `f0`, `κ`, and `κ_c`.

In [ ]:
from QickworkspaceV2.experiments.resonator import ResonatorSpec

expt = ResonatorSpec(cfg)
result = expt.run(py_avg=5)

## 4. Inspect the result

`ExperimentData` is the unified return type from every experiment.  
It supports the old tuple-unpack style **and** a rich named dict.

In [ ]:
from QickworkspaceV2 import QualityFlag

# New API
print('Experiment   :', result.experiment_type)
print('Quality      :', result.quality)
print('Fit result   :', result.fit_result)
print('Scalar result:', result.scalar_result)
print('x_axis shape :', result.x_axis.shape)
print('y_axis       :', result.y_axis.shape if result.y_axis is not None else 'None (1-D experiment)')

# Backward-compat tuple unpack
fit_params, fit_errors = result
print('fit_params   :', fit_params)

# Backward-compat scalar coercion
freq = float(result)
print(f'Resonator freq (scalar): {freq:.3f}')

## 5. Quick plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(7, 3))
plt.plot(result.x_axis, np.abs(result.raw_iq), lw=1.5)
plt.xlabel('Frequency (MHz)')
plt.ylabel('|S21| (a.u.)')
plt.title('Resonator Spectroscopy')
plt.tight_layout()


**Next:** [01_config_and_store.ipynb](01_config_and_store.ipynb) — managing experiment configuration and persisting calibration parameters.